In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, 
                            f1_score, accuracy_score, precision_score, 
                            precision_recall_curve, recall_score, confusion_matrix)
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline

# Load the dataset
data = pd.read_excel(# enter file path)

# Prepare data with updated feature selection
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome
feature_indexes = [28, 234, 36, 215, 189, 37, 77, 26, 250, 188, 236, 181, 70, 55, 48, 67, 202, 244, 78, 65, 52, 237, 211, 57, 185, 229, 231, 219, 62, 220, 51, 200, 30, 242, 233, 198, 34, 25, 212, 205, 252, 32, 29, 248, 253, 75, 66, 247, 56, 58, 12, 221, 41, 197, 7, 63, 217, 1, 251, 199, 114, 49, 24, 50, 101, 80, 186, 232, 207, 130, 79, 115, 43, 243, 190, 137, 136, 108, 46, 214, 17, 10, 110, 88, 125, 182, 33, 203, 225, 68, 213, 227, 126, 93, 81, 14, 13, 104, 92, 42, 201, 129, 177, 122, 134, 133, 60, 19, 31, 9, 135, 98, 8, 45, 3, 47, 4, 6, 226, 116, 106, 90, 15, 105, 138, 18, 89, 84, 100, 44, 228, 131, 38, 112, 103, 96, 16, 127, 206, 117, 11, 102, 176, 128, 97, 0, 132, 5, 256, 61, 235, 87, 91, 193, 39, 111, 64, 180, 99]  # Updated Relief-selected features
X = X.iloc[:, feature_indexes]

# Define the base decision tree estimator
base_estimator = DecisionTreeClassifier(
    max_depth=20,  # Increased from 15 to 20
    max_features='sqrt',  # Maintained
    min_samples_leaf=2,  # Maintained
    min_samples_split=15,  # Increased from 3 to 15
    random_state=42
)

# Define the AdaBoost classifier with updated parameters
classifier = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=250,  # Reduced from 400 to 250
    learning_rate=1e-05,  # Decreased from 0.001 to 1e-05
    algorithm='SAMME',  # Maintained
    random_state=42
)

# Define pipeline with modified ADASYN sampling
pipeline = Pipeline([
    ('sampler', ADASYN(sampling_strategy=0.2, random_state=42)),  # Reduced from 0.5 to 0.2
    ('classifier', classifier)
])

# Define pipeline with ADASYN
pipeline = Pipeline([
    ('sampler', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('classifier', classifier)
])

# Initialize cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Storage variables
all_actuals = []
all_probs = []
auc_scores = []
auprc_scores = []

# Cross-validation loop
for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probability of positive class
    
    # Store results for combined threshold calculation
    all_actuals.extend(y_test.values)
    all_probs.extend(y_prob)
    
    # Calculate fold metrics
    auc_scores.append(roc_auc_score(y_test, y_prob))
    auprc_scores.append(average_precision_score(y_test, y_prob))

# Find optimal threshold using all predictions
precision, recall, thresholds = precision_recall_curve(all_actuals, all_probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

# Recalculate metrics with optimal threshold
f1_list, acc_list, prec_list, sens_list, spec_list = [], [], [], [], []
for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)
    
    # Compute metrics
    f1_list.append(f1_score(y_test, y_pred))
    acc_list.append(accuracy_score(y_test, y_pred))
    prec_list.append(precision_score(y_test, y_pred, zero_division=0))
    sens_list.append(recall_score(y_test, y_pred))  # Sensitivity = Recall
    
    # Calculate specificity (TN / (TN + FP))
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp + 1e-9)  # Avoid division by zero
    spec_list.append(specificity)

# Calculate statistics
def format_metric(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

print("=== AdaBoost with ADASYN ===")
print(f"Optimal Threshold (F1-maximizing): {best_threshold:.4f}")
print(f"Average AUC: {format_metric(np.mean(auc_scores), np.std(auc_scores))}")
print(f"Average AUPRC: {format_metric(np.mean(auprc_scores), np.std(auprc_scores))}")
print(f"F1 Score: {format_metric(np.mean(f1_list), np.std(f1_list))}")
print(f"Accuracy: {format_metric(np.mean(acc_list), np.std(acc_list))}")
print(f"Precision: {format_metric(np.mean(prec_list), np.std(prec_list))}")
print(f"Sensitivity (Recall): {format_metric(np.mean(sens_list), np.std(sens_list))}")
print(f"Specificity: {format_metric(np.mean(spec_list), np.std(spec_list))}")

C:\Users\Inspiron\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
C:\Users\Inspiron\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


=== AdaBoost with ADASYN ===
Optimal Threshold (F1-maximizing): 0.2399
Average AUC: 0.7668 ± 0.0262
Average AUPRC: 0.2664 ± 0.0432
F1 Score: 0.3195 ± 0.0309
Accuracy: 0.7973 ± 0.0166
Precision: 0.2310 ± 0.0257
Sensitivity (Recall): 0.5194 ± 0.0384
Specificity: 0.8252 ± 0.0163
